In [ ]:
import importlib
import sys
from pathlib import Path

import pandas as pd
import plotly.io as pio
from IPython.display import display

# Interactive Plotly charts in Jupyter / VS Code / Cursor
pio.renderers.default = "plotly_mimetype+notebook_connected"

sys.path.insert(0, str(Path("..").resolve()))
sys.path.insert(0, str(Path(".").resolve()))

import plotter
importlib.reload(plotter)
from plotter import plot_series

from engine import latest_entry, steps_path

OUTCOMES_DIR = Path("../outcomes")


def load_outcome(**filters) -> pd.DataFrame:
    """Load steps for the latest registry match. Pass any of:
    strategy, id, folder, assets, params, start_date, end_date.
    """
    entry = latest_entry(OUTCOMES_DIR, **filters)
    #print(entry)
    return pd.read_json(steps_path(OUTCOMES_DIR, entry), lines=True)


In [ ]:
outcomes = {
    "hold_btc": load_outcome(strategy="hold", assets="BTC"),
    "buybelow_btc": load_outcome(strategy="buybelow", assets="BTC"),
}

for name, steps in outcomes.items():
    print(name)
    display(steps)


In [ ]:
plot_series(outcomes, "equity")


In [ ]:
# Market prices (from either outcome — both have BTC + ETH in `prices`)
market = outcomes["hold_btc"]
price_frames = {}
for asset in ["BTC"]:
    frame = market[["time", "prices"]].copy()
    frame["price"] = frame["prices"].map(lambda prices: float(prices[asset]))
    price_frames[asset] = frame

plot_series(price_frames, "price")
